# Task 1: Load and inspect the dataset
Load data_safe_copy.csv into a DataFrame named scores. Confirm the row count is at least 300 and print the first five rows. Run info() and verify that score is numeric.

In [106]:
import pandas as pd
import numpy as np

In [107]:
scores = pd.read_csv("data_safe_copy.csv")
scores

,student_id,cohort,module,assignment,score
0,S001,alpha,Module1,A1,78
1,S001,alpha,Module1,A2,84
2,S001,alpha,Module2,A1,79
3,S001,alpha,Module2,A2,86
4,S001,alpha,Module3,A1,81
...,...,...,...,...,...
295,S050,beta,Module1,A2,77
296,S050,beta,Module2,A1,73
297,S050,beta,Module2,A2,76
298,S050,beta,Module3,A1,75


In [108]:
len(scores) >= 300

True

In [109]:
scores.head(5)

,student_id,cohort,module,assignment,score
0,S001,alpha,Module1,A1,78
1,S001,alpha,Module1,A2,84
2,S001,alpha,Module2,A1,79
3,S001,alpha,Module2,A2,86
4,S001,alpha,Module3,A1,81


In [110]:
scores.info()

<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   student_id  300 non-null    str  
 1   cohort      300 non-null    str  
 2   module      300 non-null    str  
 3   assignment  300 non-null    str  
 4   score       300 non-null    int64
dtypes: int64(1), str(4)
memory usage: 11.8 KB


# Task 2: Build a student roster table and merge
Create a small DataFrame named roster with columns student_id and status, where status is either "active" or "inactive". Include at least 10 student IDs that are not present in the scores dataset. Merge scores with roster using a left join so all score rows remain. Confirm that missing statuses exist after the merge and count them.

In [111]:
roster = pd.DataFrame({'student_id':['S001','S050','Z001','Z002', 'Z003', 'Z004', 'Z005', 'Z006', 'Z007', 'Z008', 'Z009', 'Z010'],
                      'status': ['inactive','active','active', 'inactive', 'inactive', 'active', 'active', 'inactive', 'inactive', 'active','inactive', 'active']})

roster

,student_id,status
0,S001,inactive
1,S050,active
2,Z001,active
3,Z002,inactive
4,Z003,inactive
5,Z004,active
6,Z005,active
7,Z006,inactive
8,Z007,inactive
9,Z008,active


In [112]:
merged = pd.merge(scores, roster, how= 'left')
merged

,student_id,cohort,module,assignment,score,status
0,S001,alpha,Module1,A1,78,inactive
1,S001,alpha,Module1,A2,84,inactive
2,S001,alpha,Module2,A1,79,inactive
3,S001,alpha,Module2,A2,86,inactive
4,S001,alpha,Module3,A1,81,inactive
...,...,...,...,...,...,...
295,S050,beta,Module1,A2,77,active
296,S050,beta,Module2,A1,73,active
297,S050,beta,Module2,A2,76,active
298,S050,beta,Module3,A1,75,active


In [113]:
merged['status'].isna().any()

np.True_

In [114]:
merged['status'].isna().sum()

np.int64(288)

# Task 3: Aggregate by module and cohort
Compute the average score by cohort and module using groupby. Store the result in a DataFrame named avg_by_module. Then compute the overall average score by cohort only and compare the values to ensure the module-level averages roll up as expected.



In [115]:
avg_by_module = pd.DataFrame(merged.groupby(['cohort', 'module'])['score'].mean())
avg_by_module

score
cohort module            
alpha  Module1  77.970588
       Module2  78.382353
       Module3  79.794118
beta   Module1  78.647059
       Module2  78.411765
       Module3  80.029412
gamma  Module1  76.218750
       Module2  76.468750
       Module3  77.843750

In [116]:
average_score_by_cohort = avg_by_module.groupby('cohort').mean()
average_score_by_cohort

,score
cohort,
alpha,78.715686
beta,79.029412
gamma,76.843750


In [117]:
avg_by_cohort = merged.groupby('cohort')['score'].mean().to_frame()
avg_by_cohort

,score
cohort,
alpha,78.715686
beta,79.029412
gamma,76.843750


In [118]:
np.allclose(avg_by_cohort, average_score_by_cohort)

True

# Task 4: Reshape to a wide report
Create a wide table where rows are student_id, columns are module, and values are the average score per student per module. Use pivot_table with mean as the aggregation. Store the result in student_module_report.

Validate that the number of rows equals the number of unique students in the original dataset and that the column labels match the set of module names.

In [119]:
student_module_report =  merged.pivot_table(index= 'student_id', columns= 'module', values= 'score', aggfunc = 'mean')
student_module_report.head(9)

module,Module1,Module2,Module3
student_id,,,
S001,81.0,82.5,84.5
S002,73.5,72.0,75.0
S003,87.5,88.5,89.5
S004,68.5,67.5,69.5
S005,81.5,82.5,83.5
S006,76.0,75.0,77.0
S007,89.0,89.0,90.5
S008,71.0,70.0,72.0
S009,79.5,78.5,80.5


In [120]:
len(student_module_report) == len(merged['student_id'].unique())

True

In [121]:
set(merged['module']) ==set( student_module_report.columns)

True

# Task 5: Ranking and top performers
For each cohort, identify the top 3 students by overall average score. Use groupby plus sorting or rank logic. Store the results in a DataFrame named top_students with columns student_id, cohort, and avg_score.

Add a check that each cohort appears exactly three times in top_students.



In [122]:
merged

,student_id,cohort,module,assignment,score,status
0,S001,alpha,Module1,A1,78,inactive
1,S001,alpha,Module1,A2,84,inactive
2,S001,alpha,Module2,A1,79,inactive
3,S001,alpha,Module2,A2,86,inactive
4,S001,alpha,Module3,A1,81,inactive
...,...,...,...,...,...,...
295,S050,beta,Module1,A2,77,active
296,S050,beta,Module2,A1,73,active
297,S050,beta,Module2,A2,76,active
298,S050,beta,Module3,A1,75,active


In [123]:
top_students = pd.DataFrame(merged.groupby('cohort')['score'].mean().sort_values(ascending=False).head(3))
top_students

,score
cohort,
beta,79.029412
alpha,78.715686
gamma,76.843750


In [124]:
avg_scores = (
    merged
    .groupby(['cohort', 'student_id'])['score']
    .mean()
    .reset_index(name='avg_score')
)
avg_scores

,cohort,student_id,avg_score
0,alpha,S001,82.666667
1,alpha,S002,73.500000
2,alpha,S003,88.500000
3,alpha,S004,68.500000
4,alpha,S005,82.500000
5,alpha,S006,76.000000
6,alpha,S019,84.500000
7,alpha,S020,71.500000
8,alpha,S025,79.500000
9,alpha,S026,70.500000


In [125]:
top_students = (
    avg_scores
    .sort_values(['cohort', 'avg_score'], ascending=[True, False])
    .groupby('cohort')
    .head(3)
)

top_students

,cohort,student_id,avg_score
2,alpha,S003,88.5
16,alpha,S049,88.5
10,alpha,S027,86.5
31,beta,S045,90.5
17,beta,S007,89.5
23,beta,S021,88.5
35,gamma,S014,92.5
45,gamma,S040,84.5
37,gamma,S016,84.0


In [126]:
counts = top_students['cohort'].value_counts()
counts
(counts == 3).all()


np.True_